In [ ]:
#!/usr/bin/env python3

# Automated Web Scanner: HTTP Security Headers & CORS Vulnerability Auditor
**Author:** Palakpreet kaur  
**Project Name:** Automated Web Scanner  
**Description:** An automated cybersecurity auditing tool and data analysis pipeline in Python that evaluates web endpoints against OWASP Secure Headers guidelines and tests for Cross-Origin Resource Sharing (CORS) misconfigurations.

In [ ]:
import argparse
import csv
import json
import os
import re
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from typing import Dict, List, Optional
from urllib.parse import urlparse

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Optional data science & reporting libraries for Notebook analysis and .docx/.ipynb export
try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

## 1. Data Models & Benchmark Dataset Configuration
Define structured dataclasses to store individual vulnerability findings and aggregate target audit reports.

In [ ]:
DEFAULT_BENCHMARK_DOMAINS = [
    "https://www.google.com",
    "https://github.com",
    "https://www.wikipedia.org",
    "https://www.cloudflare.com",
    "https://www.mozilla.org",
    "https://example.com",
    "https://www.python.org",
    "https://owasp.org",
    "https://httpbin.org",
    "https://www.w3.org",
]


@dataclass
class Finding:
    category: str       # "HEADER", "CORS", "COOKIE", "DISCLOSURE"
    severity: str       # "HIGH", "MEDIUM", "LOW", "INFO", "PASS"
    item: str           # Header or check identifier
    message: str        # Detailed diagnostic explanation
    remediation: str = ""


@dataclass
class AuditReport:
    url: str
    final_url: str = ""
    status_code: Optional[int] = None
    timestamp: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )
    score: int = 100
    grade: str = "A+"
    high_count: int = 0
    medium_count: int = 0
    low_count: int = 0
    findings: List[Finding] = field(default_factory=list)
    raw_headers: Dict[str, str] = field(default_factory=dict)
    error: Optional[str] = None

## 2. Core Scanner Engine (`WebSecurityAuditor`)
Implements HTTP session pooling, OWASP security header validation, cookie flag inspection, server banner disclosure detection, and active CORS origin reflection & null-origin testing.

In [ ]:
class WebSecurityAuditor:
    """Audits target URLs for HTTP security headers, cookies, and CORS rules."""

    SEVERITY_WEIGHTS = {
        "HIGH": 20,
        "MEDIUM": 10,
        "LOW": 5,
        "INFO": 0,
        "PASS": 0,
    }

    SAFE_REFERRER_POLICIES = {
        "no-referrer",
        "same-origin",
        "strict-origin",
        "strict-origin-when-cross-origin",
    }

    DISCLOSURE_HEADERS = [
        "Server",
        "X-Powered-By",
        "X-AspNet-Version",
        "X-AspNetMvc-Version",
        "X-Generator",
    ]

    def __init__(
        self,
        timeout: int = 10,
        test_origin: str = "https://untrusted-auditor.example.com",
        verify_ssl: bool = True,
    ):
        self.timeout = timeout
        self.test_origin = test_origin
        self.verify_ssl = verify_ssl
        self.session = self._build_session()

    def _build_session(self) -> requests.Session:
        session = requests.Session()
        retries = Retry(
            total=2,
            backoff_factor=0.5,
            status_forcelist=[429, 500, 502, 503, 504],
        )
        adapter = HTTPAdapter(max_retries=retries)
        session.mount("http://", adapter)
        session.mount("https://", adapter)
        session.headers.update({
            "User-Agent": "AutomatedWebScanner-OWASP-Auditor/2.0"
        })
        return session

    @staticmethod
    def _normalize_url(url: str) -> str:
        url = url.strip()
        if not urlparse(url).scheme:
            return f"https://{url}"
        return url

    def audit_url(self, target_url: str) -> AuditReport:
        url = self._normalize_url(target_url)
        report = AuditReport(url=url)

        try:
            response = self.session.get(
                url,
                timeout=self.timeout,
                allow_redirects=True,
                verify=self.verify_ssl,
            )
            report.final_url = response.url
            report.status_code = response.status_code
            report.raw_headers = dict(response.headers)

            # Execute security checks
            self._check_https_enforcement(report, response)
            self._check_hsts(report, response.headers)
            self._check_csp(report, response.headers)
            self._check_frame_options(report, response.headers)
            self._check_content_type_options(report, response.headers)
            self._check_referrer_policy(report, response.headers)
            self._check_permissions_policy(report, response.headers)
            self._check_cross_origin_isolation(report, response.headers)
            self._check_info_disclosure(report, response.headers)
            self._check_cookies(report, response)
            self._check_cors(report, url)

            # Calculate compliance score & grade
            self._calculate_grade(report)

        except requests.exceptions.SSLError as exc:
            report.error = f"SSL/TLS verification failed: {exc}"
            report.score = 0
            report.grade = "F"
        except requests.RequestException as exc:
            report.error = f"HTTP request failed: {exc}"
            report.score = 0
            report.grade = "F"

        return report

    def _check_https_enforcement(
        self, report: AuditReport, response: requests.Response
    ) -> None:
        parsed = urlparse(response.url)
        if parsed.scheme != "https":
            report.findings.append(Finding(
                category="HEADER",
                severity="HIGH",
                item="HTTPS Enforcement",
                message="Final destination URL does not enforce HTTPS.",
                remediation="Redirect all HTTP requests (HTTP 301) to HTTPS.",
            ))
        else:
            report.findings.append(Finding(
                category="HEADER",
                severity="PASS",
                item="HTTPS Enforcement",
                message="Endpoint is served securely over HTTPS.",
            ))

    def _check_hsts(self, report: AuditReport, headers: Dict[str, str]) -> None:
        hsts = headers.get("Strict-Transport-Security")
        if not hsts:
            report.findings.append(Finding(
                category="HEADER",
                severity="HIGH",
                item="Strict-Transport-Security",
                message="HSTS header is missing.",
                remediation=(
                    "Add 'Strict-Transport-Security: max-age=31536000; "
                    "includeSubDomains; preload'."
                ),
            ))
            return

        match = re.search(r"max-age=(\d+)", hsts, re.IGNORECASE)
        if not match:
            report.findings.append(Finding(
                category="HEADER",
                severity="MEDIUM",
                item="Strict-Transport-Security",
                message=f"HSTS present ('{hsts}') but missing max-age directive.",
                remediation="Specify 'max-age=31536000' (at least 1 year).",
            ))
            return

        max_age = int(match.group(1))
        if max_age < 31536000:
            report.findings.append(Finding(
                category="HEADER",
                severity="LOW",
                item="Strict-Transport-Security",
                message=f"HSTS max-age ({max_age}s) is shorter than 1 year (31536000s).",
                remediation="Increase max-age to at least 31536000 seconds.",
            ))
        else:
            report.findings.append(Finding(
                category="HEADER",
                severity="PASS",
                item="Strict-Transport-Security",
                message=f"Strong HSTS policy configured ({hsts}).",
            ))

    def _check_csp(self, report: AuditReport, headers: Dict[str, str]) -> None:
        csp = headers.get("Content-Security-Policy")
        if not csp:
            report.findings.append(Finding(
                category="HEADER",
                severity="HIGH",
                item="Content-Security-Policy",
                message="Content-Security-Policy (CSP) header is missing.",
                remediation=(
                    "Define a strict CSP restricting script-src, object-src, "
                    "and base-uri."
                ),
            ))
            return

        weaknesses = []
        if "'unsafe-inline'" in csp:
            weaknesses.append("uses 'unsafe-inline'")
        if "'unsafe-eval'" in csp:
            weaknesses.append("uses 'unsafe-eval'")
        if re.search(r"(default-src|script-src)[^;]*\*", csp):
            weaknesses.append("allows wildcard (*) sources in script/default-src")
        if "default-src" not in csp and "script-src" not in csp:
            weaknesses.append("missing fallback 'default-src' and 'script-src'")

        if weaknesses:
            report.findings.append(Finding(
                category="HEADER",
                severity="MEDIUM",
                item="Content-Security-Policy",
                message=f"CSP is present but contains weaknesses: {', '.join(weaknesses)}.",
                remediation=(
                    "Replace 'unsafe-inline' with nonces or hashes, and remove "
                    "'unsafe-eval' and wildcard script sources."
                ),
            ))
        else:
            report.findings.append(Finding(
                category="HEADER",
                severity="PASS",
                item="Content-Security-Policy",
                message="CSP header is present without common unsafe directives.",
            ))

    def _check_frame_options(
        self, report: AuditReport, headers: Dict[str, str]
    ) -> None:
        xfo = headers.get("X-Frame-Options", "").upper()
        csp = headers.get("Content-Security-Policy", "")
        has_frame_ancestors = "frame-ancestors" in csp.lower()

        if not xfo and not has_frame_ancestors:
            report.findings.append(Finding(
                category="HEADER",
                severity="MEDIUM",
                item="X-Frame-Options",
                message="Neither X-Frame-Options nor CSP 'frame-ancestors' is set.",
                remediation="Set 'X-Frame-Options: DENY' or 'SAMEORIGIN'.",
            ))
        elif xfo and xfo not in {"DENY", "SAMEORIGIN"} and not has_frame_ancestors:
            report.findings.append(Finding(
                category="HEADER",
                severity="LOW",
                item="X-Frame-Options",
                message=f"Non-standard X-Frame-Options value: '{xfo}'.",
                remediation="Use 'DENY' or 'SAMEORIGIN'.",
            ))
        else:
            report.findings.append(Finding(
                category="HEADER",
                severity="PASS",
                item="X-Frame-Options",
                message="Clickjacking protection is properly configured.",
            ))

    def _check_content_type_options(
        self, report: AuditReport, headers: Dict[str, str]
    ) -> None:
        xcto = headers.get("X-Content-Type-Options", "").lower()
        if xcto != "nosniff":
            report.findings.append(Finding(
                category="HEADER",
                severity="MEDIUM",
                item="X-Content-Type-Options",
                message="X-Content-Type-Options is missing or not set to 'nosniff'.",
                remediation="Set 'X-Content-Type-Options: nosniff'.",
            ))
        else:
            report.findings.append(Finding(
                category="HEADER",
                severity="PASS",
                item="X-Content-Type-Options",
                message="X-Content-Type-Options is set to 'nosniff'.",
            ))

    def _check_referrer_policy(
        self, report: AuditReport, headers: Dict[str, str]
    ) -> None:
        ref = headers.get("Referrer-Policy", "").lower()
        if not ref:
            report.findings.append(Finding(
                category="HEADER",
                severity="LOW",
                item="Referrer-Policy",
                message="Referrer-Policy header is missing.",
                remediation=(
                    "Set 'Referrer-Policy: strict-origin-when-cross-origin' "
                    "or 'no-referrer'."
                ),
            ))
            return

        policies = [p.strip() for p in ref.split(",") if p.strip()]
        effective_policy = policies[-1] if policies else ""

        if effective_policy not in self.SAFE_REFERRER_POLICIES:
            report.findings.append(Finding(
                category="HEADER",
                severity="LOW",
                item="Referrer-Policy",
                message=f"Referrer-Policy ('{ref}') may leak full URLs cross-origin.",
                remediation="Use 'strict-origin-when-cross-origin' or 'no-referrer'.",
            ))
        else:
            report.findings.append(Finding(
                category="HEADER",
                severity="PASS",
                item="Referrer-Policy",
                message=f"Privacy-preserving Referrer-Policy configured ('{ref}').",
            ))

    def _check_permissions_policy(
        self, report: AuditReport, headers: Dict[str, str]
    ) -> None:
        pp = headers.get("Permissions-Policy")
        if not pp:
            report.findings.append(Finding(
                category="HEADER",
                severity="LOW",
                item="Permissions-Policy",
                message="Permissions-Policy header is missing.",
                remediation=(
                    "Add Permissions-Policy to restrict sensitive browser features "
                    "(e.g., 'geolocation=(), microphone=(), camera=()')."
                ),
            ))
        else:
            report.findings.append(Finding(
                category="HEADER",
                severity="PASS",
                item="Permissions-Policy",
                message="Permissions-Policy header is present.",
            ))

    def _check_cross_origin_isolation(
        self, report: AuditReport, headers: Dict[str, str]
    ) -> None:
        coop = headers.get("Cross-Origin-Opener-Policy")
        corp = headers.get("Cross-Origin-Resource-Policy")
        if not coop or not corp:
            missing = [
                name for name, val in [
                    ("Cross-Origin-Opener-Policy", coop),
                    ("Cross-Origin-Resource-Policy", corp),
                ] if not val
            ]
            report.findings.append(Finding(
                category="HEADER",
                severity="INFO",
                item="Cross-Origin Isolation",
                message=f"Optional isolation headers not set: {', '.join(missing)}.",
                remediation="Consider setting COOP and CORP to 'same-origin'.",
            ))
        else:
            report.findings.append(Finding(
                category="HEADER",
                severity="PASS",
                item="Cross-Origin Isolation",
                message=f"COOP ('{coop}') and CORP ('{corp}') are configured.",
            ))

    def _check_info_disclosure(
        self, report: AuditReport, headers: Dict[str, str]
    ) -> None:
        for h_name in self.DISCLOSURE_HEADERS:
            val = headers.get(h_name)
            if val:
                has_version = bool(re.search(r"\d+\.\d+", val))
                severity = "LOW" if has_version else "INFO"
                report.findings.append(Finding(
                    category="DISCLOSURE",
                    severity=severity,
                    item=h_name,
                    message=f"Header '{h_name}: {val}' exposes server technology metadata.",
                    remediation=f"Remove or obfuscate the '{h_name}' response header.",
                ))

    def _check_cookies(
        self, report: AuditReport, response: requests.Response
    ) -> None:
        raw_cookies = (
            response.raw.headers.getlist("Set-Cookie")
            if hasattr(response.raw, "headers")
            and hasattr(response.raw.headers, "getlist")
            else []
        )
        if not raw_cookies and "Set-Cookie" in response.headers:
            raw_cookies = [response.headers["Set-Cookie"]]

        for cookie_str in raw_cookies:
            cookie_name = cookie_str.split("=", 1)[0].strip()
            lower_cookie = cookie_str.lower()
            missing_flags = []

            if "secure" not in lower_cookie:
                missing_flags.append("Secure")
            if "httponly" not in lower_cookie:
                missing_flags.append("HttpOnly")
            if "samesite" not in lower_cookie:
                missing_flags.append("SameSite")

            if missing_flags:
                report.findings.append(Finding(
                    category="COOKIE",
                    severity="MEDIUM",
                    item=f"Cookie: {cookie_name}",
                    message=(
                        f"Cookie '{cookie_name}' is missing security flags: "
                        f"{', '.join(missing_flags)}."
                    ),
                    remediation="Set 'Secure; HttpOnly; SameSite=Lax' (or Strict).",
                ))
            else:
                report.findings.append(Finding(
                    category="COOKIE",
                    severity="PASS",
                    item=f"Cookie: {cookie_name}",
                    message=f"Cookie '{cookie_name}' has Secure, HttpOnly, and SameSite.",
                ))

    def _check_cors(self, report: AuditReport, url: str) -> None:
        """Tests endpoint against arbitrary Origin reflection and null Origin bypasses."""
        try:
            # Test 1: Arbitrary untrusted origin
            cors_resp = self.session.get(
                url,
                headers={"Origin": self.test_origin},
                timeout=self.timeout,
                allow_redirects=True,
                verify=self.verify_ssl,
            )
            acao = cors_resp.headers.get("Access-Control-Allow-Origin")
            acac = (
                cors_resp.headers.get("Access-Control-Allow-Credentials", "").lower()
                == "true"
            )
            vary = cors_resp.headers.get("Vary", "")

            # Test 2: Null origin probe
            null_resp = self.session.get(
                url,
                headers={"Origin": "null"},
                timeout=self.timeout,
                allow_redirects=True,
                verify=self.verify_ssl,
            )
            null_acao = null_resp.headers.get("Access-Control-Allow-Origin")
            null_acac = (
                null_resp.headers.get("Access-Control-Allow-Credentials", "").lower()
                == "true"
            )

            if not acao and not null_acao:
                report.findings.append(Finding(
                    category="CORS",
                    severity="PASS",
                    item="CORS Policy",
                    message="No permissive CORS headers returned for untrusted origins.",
                ))
                return

            if acao == "*" and acac:
                report.findings.append(Finding(
                    category="CORS",
                    severity="HIGH",
                    item="CORS Wildcard + Credentials",
                    message="Access-Control-Allow-Origin is '*' with Allow-Credentials: true.",
                    remediation="Never combine wildcard origins with Allow-Credentials: true.",
                ))
            elif acao == self.test_origin and acac:
                report.findings.append(Finding(
                    category="CORS",
                    severity="HIGH",
                    item="CORS Origin Reflection (Credentialed)",
                    message=(
                        f"Untrusted origin ('{self.test_origin}') reflected in ACAO "
                        "with Allow-Credentials: true."
                    ),
                    remediation="Validate Origin against a strict server-side allowlist.",
                ))
            elif acao == self.test_origin:
                report.findings.append(Finding(
                    category="CORS",
                    severity="MEDIUM",
                    item="CORS Origin Reflection",
                    message=f"Untrusted origin ('{self.test_origin}') reflected in ACAO.",
                    remediation="Restrict allowed origins to an explicit trusted allowlist.",
                ))
            elif acao == "*":
                report.findings.append(Finding(
                    category="CORS",
                    severity="INFO",
                    item="CORS Public Wildcard",
                    message="Endpoint allows public cross-origin access ('*').",
                    remediation="Ensure endpoint serves only public, non-sensitive data.",
                ))

            if null_acao == "null":
                sev = "HIGH" if null_acac else "MEDIUM"
                report.findings.append(Finding(
                    category="CORS",
                    severity=sev,
                    item="CORS Null Origin Allowed",
                    message="Endpoint permits 'Origin: null', exploitable via sandboxed iframes.",
                    remediation="Reject 'null' origins in CORS configuration.",
                ))

            if acao not in {"*", None} and "origin" not in vary.lower():
                report.findings.append(Finding(
                    category="CORS",
                    severity="LOW",
                    item="CORS Cache Header (Vary: Origin)",
                    message="Dynamic ACAO returned without 'Vary: Origin' header.",
                    remediation="Include 'Vary: Origin' when dynamically setting ACAO.",
                ))

        except requests.RequestException as exc:
            report.findings.append(Finding(
                category="CORS",
                severity="INFO",
                item="CORS Check",
                message=f"Could not complete CORS probe: {exc}",
            ))

    def _calculate_grade(self, report: AuditReport) -> None:
        report.high_count = sum(1 for f in report.findings if f.severity == "HIGH")
        report.medium_count = sum(1 for f in report.findings if f.severity == "MEDIUM")
        report.low_count = sum(1 for f in report.findings if f.severity == "LOW")

        deductions = sum(
            self.SEVERITY_WEIGHTS.get(f.severity, 0) for f in report.findings
        )
        report.score = max(0, 100 - deductions)

        if report.score >= 95:
            report.grade = "A+"
        elif report.score >= 90:
            report.grade = "A"
        elif report.score >= 80:
            report.grade = "B"
        elif report.score >= 70:
            report.grade = "C"
        elif report.score >= 60:
            report.grade = "D"
        else:
            report.grade = "F"

## 3. Dataset Analytics & Visualization Pipeline
Converts scan results into a structured Pandas DataFrame, exports CSV/JSON summaries, and generates visual charts for the Project Report.

In [ ]:
def generate_analytics_and_charts(
    reports: List[AuditReport], output_prefix: str = "scan_results"
) -> None:
    """Exports CSV summary and creates visualizations if pandas/matplotlib are installed."""
    summary_rows = []
    for r in reports:
        missing_headers = [
            f.item for f in r.findings
            if f.category == "HEADER" and f.severity in {"HIGH", "MEDIUM", "LOW"}
        ]
        cors_issues = [
            f.item for f in r.findings
            if f.category == "CORS" and f.severity in {"HIGH", "MEDIUM", "LOW"}
        ]
        summary_rows.append({
            "URL": r.url,
            "Final_URL": r.final_url,
            "HTTP_Status": r.status_code,
            "Score": r.score,
            "Grade": r.grade,
            "High_Findings": r.high_count,
            "Medium_Findings": r.medium_count,
            "Low_Findings": r.low_count,
            "Weak_or_Missing_Headers": "; ".join(missing_headers) or "None",
            "CORS_Flaws": "; ".join(cors_issues) or "None",
        })

    csv_path = f"{output_prefix}.csv"
    if summary_rows:
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
            writer.writeheader()
            writer.writerows(summary_rows)
        print(f"[+] Summary CSV saved to: {csv_path}")

    if pd is not None and summary_rows:
        df = pd.DataFrame(summary_rows)
        print("\n=== Dataset Compliance Summary Table ===")
        print(df[["URL", "HTTP_Status", "Score", "Grade", "High_Findings", "Medium_Findings"]].to_string(index=False))

    if plt is not None and summary_rows:
        grades = [r["Grade"] for r in summary_rows]
        grade_order = ["A+", "A", "B", "C", "D", "F"]
        counts = [grades.count(g) for g in grade_order]

        plt.figure(figsize=(8, 5))
        plt.bar(grade_order, counts, color=["#1b9e77", "#66a61e", "#7570b3", "#e6ab02", "#d95f02", "#e7298a"])
        plt.title("Security Header & CORS Compliance Grade Distribution")
        plt.xlabel("Security Grade")
        plt.ylabel("Number of Audited Websites")
        plt.grid(axis="y", linestyle="--", alpha=0.6)
        chart_path = f"{output_prefix}_grades.png"
        plt.tight_layout()
        plt.savefig(chart_path, dpi=150)
        plt.close()
        print(f"[+] Grade distribution chart saved to: {chart_path}")

## 4. Automated Submission File Generator (`.ipynb` and `.docx`)
Running this script with `--generate-submission` automatically builds `YourName_ProjectName.ipynb` and `YourName_ProjectReport.docx` in the current directory.

In [ ]:
def export_submission_files() -> None:
    """Generates YourName_ProjectName.ipynb and YourName_ProjectReport.docx."""
    # 1. Generate .ipynb using nbformat
    try:
        import nbformat as nbf
        nb = nbf.v4.new_notebook()
        with open(__file__, "r", encoding="utf-8") as f:
            source_code = f.read()

        raw_cells = re.split(r"^# %%(?: \[(markdown)\])?\s*$", source_code, flags=re.MULTILINE)
        # raw_cells[0] is preamble; then pairs of (cell_type, content)
        cells = []
        idx = 1
        while idx < len(raw_cells):
            cell_type = raw_cells[idx]
            content = raw_cells[idx + 1].strip()
            if cell_type == "markdown":
                md_lines = [
                     re.sub(r"^# ?", "", line) for line in content.splitlines()
                ]
                cells.append(nbf.v4.new_markdown_cell("\n".join(md_lines)))
            else:
                cells.append(nbf.v4.new_code_cell(content))
            idx += 2

        nb["cells"] = cells
        ipynb_name = "YourName_ProjectName.ipynb"
        with open(ipynb_name, "w", encoding="utf-8") as f:
            nbf.write(nb, f)
        print(f"[+] Successfully generated Jupyter Notebook: {ipynb_name}")
    except ImportError:
        print("[!] Install 'nbformat' (pip install nbformat) to auto-generate the .ipynb file.")

    # 2. Generate .docx Project Report using python-docx
    try:
        from docx import Document
        doc = Document()
        doc.add_heading("Automated Web Scanner: HTTP Security Header & CORS Auditor", 0)
        doc.add_paragraph("Author: Palakpreet Kaur | Project Submission Report")

        report_md = "PalakpreetKaur_AutomatedWebScanner.md"
        if os.path.exists(report_md):
            with open(report_md, "r", encoding="utf-8") as f:
                for line in f:
                    stripped = line.strip()
                    if stripped.startswith("# "):
                        doc.add_heading(stripped[2:], level=1)
                    elif stripped.startswith("## "):
                        doc.add_heading(stripped[3:], level=2)
                    elif stripped.startswith("### "):
                        doc.add_heading(stripped[4:], level=3)
                    elif stripped.startswith("- ") or stripped.startswith("* "):
                        doc.add_paragraph(stripped[2:], style="List Bullet")
                    elif stripped:
                        doc.add_paragraph(stripped)
        else:
            doc.add_heading("1. Executive Summary", level=1)
            doc.add_paragraph(
                "This project implements an automated Python security scanner to audit "
                "websites against OWASP Secure Headers guidelines and detect permissive "
                "Cross-Origin Resource Sharing (CORS) configurations."
            )

        docx_name = "PalakpreetKaur_AutomatedWebScanner.docx"
        doc.save(docx_name)
        print(f"[+] Successfully generated Word Project Report: {docx_name}")
    except ImportError:
        print("[!] Install 'python-docx' (pip install python-docx) to auto-generate the .docx file.")

## 5. Command-Line Interface & Execution Entrypoint

In [ ]:
def print_console_report(report: AuditReport) -> None:
    divider = "=" * 72
    print(f"\n{divider}")
    print(f"TARGET: {report.url}")
    if report.error:
        print(f"STATUS: ERROR -> {report.error}")
        print(divider)
        return

    print(f"FINAL URL: {report.final_url} (HTTP {report.status_code})")
    print(f"SECURITY SCORE: {report.score}/100  |  GRADE: {report.grade}")
    print(divider)

    severity_order = {"HIGH": 1, "MEDIUM": 2, "LOW": 3, "INFO": 4, "PASS": 5}
    sorted_findings = sorted(
        report.findings, key=lambda x: severity_order.get(x.severity, 99)
    )

    for finding in sorted_findings:
        tag = f"[{finding.severity}]".ljust(8)
        print(f"{tag} ({finding.category}) {finding.item}: {finding.message}")
        if finding.remediation and finding.severity in {"HIGH", "MEDIUM", "LOW"}:
            print(f"         -> Fix: {finding.remediation}")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Automated Web Security Header & CORS Compliance Scanner"
    )
    parser.add_argument(
        "urls",
        nargs="*",
        help="Target URLs to audit (defaults to built-in benchmark dataset if omitted)",
    )
    parser.add_argument(
        "-f", "--file",
        help="Path to text/CSV file containing target URLs (one per line)",
    )
    parser.add_argument(
        "-o", "--output",
        default="scan_results.json",
        help="Output JSON report path (default: scan_results.json)",
    )
    parser.add_argument(
        "-w", "--workers",
        type=int,
        default=5,
        help="Concurrent worker threads (default: 5)",
    )
    parser.add_argument(
        "-t", "--timeout",
        type=int,
        default=10,
        help="HTTP timeout in seconds (default: 10)",
    )
    parser.add_argument(
        "--generate-submission",
        action="store_true",
        help="Automatically build YourName_ProjectName.ipynb and YourName_ProjectReport.docx",
    )
    return parser.parse_args()


def main() -> int:
    # Ignore Jupyter kernel arguments if executed inside an interactive notebook
    if "ipykernel" in sys.modules:
        targets = DEFAULT_BENCHMARK_DOMAINS[:5]
        auditor = WebSecurityAuditor()
        reports = [auditor.audit_url(u) for u in targets]
        for r in reports:
            print_console_report(r)
        generate_analytics_and_charts(reports)
        return 0

    args = parse_args()

    if args.generate_submission:
        export_submission_files()
        return 0

    targets: List[str] = list(args.urls)
    if args.file:
        try:
            with open(args.file, "r", encoding="utf-8") as fh:
                for line in fh:
                    clean = line.strip()
                    if clean and not clean.startswith("#"):
                        # Support Tranco CSV format (rank,domain) or plain URL list
                        parts = clean.split(",")
                        domain = parts[-1].strip()
                        if domain.lower() != "domain" and domain.lower() != "url":
                            targets.append(domain)
        except OSError as exc:
            print(f"[!] Error reading file '{args.file}': {exc}", file=sys.stderr)
            return 1

    if not targets:
        print("[*] No URLs specified via CLI; auditing built-in benchmark dataset...")
        targets = DEFAULT_BENCHMARK_DOMAINS

    auditor = WebSecurityAuditor(timeout=args.timeout)
    reports: List[AuditReport] = []

    with ThreadPoolExecutor(max_workers=args.workers) as executor:
        future_to_url = {
            executor.submit(auditor.audit_url, url): url for url in targets
        }
        for future in as_completed(future_to_url):
            report = future.result()
            reports.append(report)
            print_console_report(report)

    # Save structured JSON output
    if args.output:
        with open(args.output, "w", encoding="utf-8") as out_fh:
            json.dump([asdict(r) for r in reports], out_fh, indent=2)
        print(f"\n[+] Full JSON audit report saved to: {args.output}")

    # Run analytics table & chart export
    generate_analytics_and_charts(reports)
    return 0


if __name__ == "__main__":
    sys.exit(main())